# NHANES diabetes project — Steps 1 to 5

1. Download NHANES files (2011–12 … 2021–23)
2. Merge on SEQN, clean special codes
3. Survey design columns + derived variables
4. Survey-weighted prevalence with design-based SEs (validate against NCHS)
5. Undiagnosed case counts per cycle × subgroup

Run the cells top to bottom. Outputs go to `nhanes_out/`.

## 0. Setup

In [ ]:
import io
import os

import numpy as np
import pandas as pd
import requests

RAW_DIR = "nhanes_raw"   # downloaded .xpt files are cached here
OUT_DIR = "nhanes_out"   # merged data + tables are written here
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# cycle -> (begin year used in CDC URLs, file-name prefix, file-name suffix, legacy folder)
CYCLES = {
    "2011-2012": (2011, "",   "_G", "2011-2012"),
    "2013-2014": (2013, "",   "_H", "2013-2014"),
    "2015-2016": (2015, "",   "_I", "2015-2016"),
    "2017-2020": (2017, "P_", "",   "2017-2018"),   # pre-pandemic file (P_ prefix)
    "2021-2023": (2021, "",   "_L", "2021-2022"),
}

# file -> columns we want (weights are picked up automatically: any column starting with "WT")
FILES = {
    "DEMO": ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "INDFMPIR",
             "RIDEXPRG", "SDMVSTRA", "SDMVPSU"],
    "DIQ":  ["SEQN", "DIQ010", "DIQ160", "DIQ180", "DIQ050", "DIQ070"],
    "HIQ":  ["SEQN", "HIQ011"],
    "HUQ":  ["SEQN", "HUQ030"],
    "BMX":  ["SEQN", "BMXBMI", "BMXWAIST"],
    "GHB":  ["SEQN", "LBXGH"],
    "GLU":  ["SEQN", "LBXGLU"],
}

HEADERS = {"User-Agent": "Mozilla/5.0 (research download script)"}

## 1. Download

In [ ]:
def file_name(base, cycle):
    _, prefix, suffix, _ = CYCLES[cycle]
    return f"{prefix}{base}{suffix}"


def candidate_urls(name, cycle):
    year, _, _, legacy = CYCLES[cycle]
    return [
        f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}.xpt",
        f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}.XPT",
        f"https://wwwn.cdc.gov/Nchs/Nhanes/{legacy}/{name}.XPT",
    ]


def download(base, cycle):
    """Download one NHANES .xpt file (cached). Returns local path or None."""
    name = file_name(base, cycle)
    path = os.path.join(RAW_DIR, f"{name}.xpt")
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    for url in candidate_urls(name, cycle):
        try:
            r = requests.get(url, headers=HEADERS, timeout=60)
        except requests.RequestException:
            continue
        # a real SAS transport file starts with "HEADER RECORD"; anything else is an error page
        if r.status_code == 200 and r.content[:13] == b"HEADER RECORD":
            with open(path, "wb") as f:
                f.write(r.content)
            return path
    print(f"  !! could not download {name} ({cycle}) — copy the link from the NHANES data page")
    return None


def read_xpt(path, wanted):
    df = pd.read_sas(path, format="xport", encoding="latin-1")
    df.columns = [c.upper() for c in df.columns]
    keep = [c for c in wanted if c in df.columns] + [c for c in df.columns if c.startswith("WT")]
    return df[list(dict.fromkeys(keep))]


raw = {}  # (base, cycle) -> DataFrame
for cycle in CYCLES:
    print(f"Downloading {cycle} ...")
    for base, cols in FILES.items():
        p = download(base, cycle)
        if p:
            raw[(base, cycle)] = read_xpt(p, cols)
print("Done.")

## 2. Merge, clean codes, harmonize

In [ ]:
def merge_cycle(cycle):
    if ("DEMO", cycle) not in raw:
        raise RuntimeError(f"DEMO missing for {cycle}; cannot build this cycle")
    df = raw[("DEMO", cycle)].copy()
    for base in FILES:
        if base == "DEMO" or (base, cycle) not in raw:
            continue
        other = raw[(base, cycle)]
        # don't duplicate weight columns that already exist (e.g., none expected, but be safe)
        other = other[[c for c in other.columns if c == "SEQN" or c not in df.columns]]
        df = df.merge(other, on="SEQN", how="left")
    df["cycle"] = cycle
    return df


df = pd.concat([merge_cycle(c) for c in CYCLES if ("DEMO", c) in raw], ignore_index=True)
df["SEQN"] = df["SEQN"].astype("int64")

# --- variable availability report: check this BEFORE trusting any comparison across cycles
check_vars = ["DIQ010", "DIQ180", "HIQ011", "HUQ030", "LBXGH", "LBXGLU", "BMXBMI", "INDFMPIR"]
avail = (df.groupby("cycle")[check_vars]
           .apply(lambda g: g.notna().mean().round(3))
           .reindex(columns=check_vars))
print("\nShare of non-missing values per cycle (0 = variable absent in that cycle):")
print(avail.to_string())

# --- special codes: 7/9 (and 77/99 etc.) = Refused / Don't know -> missing
SPECIAL = [7, 9, 77, 99, 777, 999, 7777, 9999]
for col in ["DIQ010", "DIQ160", "DIQ180", "DIQ050", "DIQ070", "HIQ011", "HUQ030", "DMDEDUC2"]:
    if col in df:
        df[col] = df[col].where(~df[col].isin(SPECIAL))

## 3. Survey design columns + derived variables

In [ ]:
def first_present(row_df, names):
    for n in names:
        if n in row_df.columns:
            return row_df[n]
    return pd.Series(np.nan, index=row_df.index)


# weights: 2-yr cycles use *2YR; the 2017-2020 pre-pandemic file uses *PRP
df["w_int"] = first_present(df, ["WTINT2YR"]).fillna(first_present(df, ["WTINTPRP"]))
df["w_mec"] = first_present(df, ["WTMEC2YR"]).fillna(first_present(df, ["WTMECPRP"]))
fast_cols = [c for c in df.columns if c.startswith("WTSAF")]   # fasting-subsample weight
df["w_fast"] = df[fast_cols].bfill(axis=1).iloc[:, 0] if fast_cols else np.nan

# strata / PSU numbers are only meaningful within a cycle -> make them unique across cycles
df["stratum"] = df["cycle"] + "_" + df["SDMVSTRA"].astype("Int64").astype(str)
df["psu"] = df["stratum"] + "_" + df["SDMVPSU"].astype("Int64").astype(str)

# analysis population: adults 20+, not pregnant (standard NCHS diabetes definition)
pregnant = df["RIDEXPRG"].eq(1) if "RIDEXPRG" in df else False
df["adult"] = (df["RIDAGEYR"] >= 20) & ~pregnant


def flag(cond, valid):
    """1.0 / 0.0 where valid, NaN otherwise."""
    return np.where(valid, cond.astype(float), np.nan)


# self-reported diagnosis (borderline = 3 is NOT a diagnosis)
df["diagnosed"] = df["DIQ010"].map({1: 1.0, 2: 0.0, 3: 0.0})

# biomarkers
df["a1c_pos"] = flag(df["LBXGH"] >= 6.5, df["LBXGH"].notna())
df["fpg_pos"] = flag(df["LBXGLU"] >= 126, df["LBXGLU"].notna())

# definition A (HbA1c only; MEC weight; larger sample)
ok_a = df["diagnosed"].notna() & df["a1c_pos"].notna()
df["total_dm_a"] = flag((df["diagnosed"] == 1) | (df["a1c_pos"] == 1), ok_a)
df["undx_a"] = flag((df["diagnosed"] == 0) & (df["a1c_pos"] == 1), ok_a)

# definition B (HbA1c OR fasting glucose; fasting subsample + fasting weight; NCHS-style)
ok_b = ok_a & df["fpg_pos"].notna() & (df["w_fast"].fillna(0) > 0)
bio_b = (df["a1c_pos"] == 1) | (df["fpg_pos"] == 1)
df["total_dm_b"] = flag((df["diagnosed"] == 1) | bio_b, ok_b)
df["undx_b"] = flag((df["diagnosed"] == 0) & bio_b, ok_b)

# testing (DIQ180 is only asked of people WITHOUT a diagnosis)
df["tested_3y"] = df["DIQ180"].map({1: 1.0, 2: 0.0})

# access / socioeconomic subgroups
df["insured"] = df["HIQ011"].map({1: 1.0, 2: 0.0})
df["routine_care"] = df["HUQ030"].map({1: 1.0, 3: 1.0, 2: 0.0})
df["income_grp"] = pd.cut(df["INDFMPIR"], [-np.inf, 1.3, 3.5, np.inf],
                          labels=["<1.3 PIR", "1.3-3.5 PIR", ">3.5 PIR"], right=False)
df["age_grp"] = pd.cut(df["RIDAGEYR"], [20, 40, 60, np.inf],
                       labels=["20-39", "40-59", "60+"], right=False)

KEEP_OUT = ["SEQN", "cycle", "stratum", "psu", "w_int", "w_mec", "w_fast", "adult",
            "RIDAGEYR", "RIAGENDR", "RIDRETH3", "DMDEDUC2", "INDFMPIR", "BMXBMI", "BMXWAIST",
            "LBXGH", "LBXGLU", "DIQ010", "DIQ180", "HIQ011", "HUQ030",
            "diagnosed", "a1c_pos", "fpg_pos", "total_dm_a", "undx_a", "total_dm_b", "undx_b",
            "tested_3y", "insured", "routine_care", "income_grp", "age_grp"]
clean = df[[c for c in KEEP_OUT if c in df.columns]]
clean.to_csv(os.path.join(OUT_DIR, "nhanes_clean_2011_2023.csv"), index=False)
print(f"\nSaved {len(clean):,} rows -> {OUT_DIR}/nhanes_clean_2011_2023.csv")

## 4. Survey-weighted prevalence (design-based Taylor SE)

In [ ]:
def svy_prop(d, y, w, domain=None):
    """
    Weighted proportion of binary y within a domain, for ONE cycle.
    SE by Taylor linearization (with-replacement PSU variance within strata).
    Domain rows are zeroed rather than dropped, so all PSUs stay in the design
    (this is what R's survey::svyby / subset does).
    """
    in_dom = (d[w].fillna(0) > 0) & d[y].notna()
    if domain is not None:
        in_dom &= domain
    wt = np.where(in_dom, d[w].fillna(0), 0.0)
    yv = np.where(in_dom, d[y].fillna(0), 0.0)
    W = wt.sum()
    if W == 0:
        return np.nan, np.nan, 0
    p = (wt * yv).sum() / W
    z = wt * (yv - p) / W
    psu_tot = pd.DataFrame({"h": d["stratum"].values, "psu": d["psu"].values, "z": z}) \
                .groupby(["h", "psu"], observed=True)["z"].sum().reset_index()
    var = 0.0
    for _, g in psu_tot.groupby("h"):
        n = len(g)
        if n > 1:
            var += n / (n - 1) * ((g["z"] - g["z"].mean()) ** 2).sum()
    return p, np.sqrt(var), int(in_dom.sum())


rows = []
for cycle, d in clean.groupby("cycle"):
    adult = d["adult"]
    for label, y, w in [
        ("Diagnosed (DIQ010)",           "diagnosed",  "w_mec"),
        ("Undiagnosed, HbA1c only",      "undx_a",     "w_mec"),
        ("Total, HbA1c only",            "total_dm_a", "w_mec"),
        ("Undiagnosed, HbA1c or FPG",    "undx_b",     "w_fast"),
        ("Total, HbA1c or FPG",          "total_dm_b", "w_fast"),
    ]:
        p, se, n = svy_prop(d, y, w, adult)
        rows.append({"cycle": cycle, "measure": label, "weight": w, "n": n,
                     "pct": 100 * p, "se": 100 * se,
                     "ci_low": 100 * (p - 1.96 * se), "ci_high": 100 * (p + 1.96 * se)})

prev = pd.DataFrame(rows)
prev.to_csv(os.path.join(OUT_DIR, "step4_weighted_prevalence.csv"), index=False)
print("\nStep 4 — weighted prevalence among adults 20+ (%):")
print(prev.pivot(index="measure", columns="cycle", values="pct").round(1).to_string())
print("\n>> Validate: compare 'Total/Undiagnosed, HbA1c or FPG' with the NCHS Data Brief")
print("   for the same cycle (search 'NCHS data brief diabetes prevalence NHANES').")
print("   If yours is within the published CI, the pipeline and weights are correct.")

## 5. Case counts per cycle x subgroup (power check)

In [ ]:
MIN_EVENTS = 30
count_rows = []
for cycle, d in clean[clean["adult"]].groupby("cycle"):
    base = d[d["undx_a"].notna()]
    for var in ["income_grp", "insured", "routine_care", "age_grp", "RIAGENDR"]:
        if var not in base:
            continue
        for level, g in base.groupby(var, observed=True, dropna=False):
            count_rows.append({
                "cycle": cycle, "subgroup": var, "level": level,
                "n_adults_with_data": len(g),
                "n_undiagnosed": int(g["undx_a"].sum()),
                "n_answered_DIQ180": int(g["tested_3y"].notna().sum()),
            })

counts = pd.DataFrame(count_rows)
counts["too_small"] = counts["n_undiagnosed"] < MIN_EVENTS
counts.to_csv(os.path.join(OUT_DIR, "step5_case_counts.csv"), index=False)

print(f"\nStep 5 — undiagnosed cases (HbA1c definition), unweighted counts:")
print(counts.pivot_table(index=["subgroup", "level"], columns="cycle",
                         values="n_undiagnosed", aggfunc="sum").to_string())
small = counts[counts["too_small"]]
print(f"\n{len(small)} cycle x subgroup cells have fewer than {MIN_EVENTS} undiagnosed cases.")
print("Merge those groups or pool cycles before Phase 2.")
print(f"\nAll outputs are in ./{OUT_DIR}/")